# Module 0: Environment Setup

Welcome to the **nanochat** learning guide! This notebook will help you set up your environment for the entire tutorial series.

## What You'll Learn

- Setting up the development environment on Google Colab
- Installing dependencies (PyTorch, tiktoken, etc.)
- Verifying GPU access and capabilities
- Understanding the project structure

## Prerequisites

- Google Colab account (free tier works, but A100 recommended)
- Basic familiarity with Python and Jupyter notebooks

## 0.1 Check Runtime Environment

First, let's verify we're running on a GPU and check its specifications.

In [ ]:
import torch
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {props.name}")
        print(f"  - Memory: {props.total_memory / 1024**3:.1f} GB")
        print(f"  - Compute Capability: {props.major}.{props.minor}")
        print(f"  - Multi-Processor Count: {props.multi_processor_count}")
else:
    print("\n⚠️ No GPU detected! Please enable GPU in Colab:")
    print("   Runtime → Change runtime type → Hardware accelerator → GPU")

### GPU Requirements for This Tutorial

| Notebook | Minimum | Recommended |
|----------|---------|-------------|
| 00-02 (Foundations) | T4 (16GB) | A100 (40GB) |
| 03-06 (Training) | A100 (40GB) | A100 (80GB) |
| 07-12 (Inference) | T4 (16GB) | A100 (40GB) |

For the full training pipeline, we recommend an **A100 80GB**. If you have access to Colab Pro/Pro+, select this option.

## 0.2 Clone and Install Nanochat

Let's clone the nanochat repository and install its dependencies.

In [ ]:
# Clone the repository (skip if already cloned)
import os

if not os.path.exists('nanochat'):
    !git clone https://github.com/Ivis4ml/nanochat.git
    %cd nanochat
else:
    %cd nanochat
    !git pull

print(f"\nCurrent directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -e . -q

# Verify installation
import tiktoken
import datasets
import fastapi

print("✅ Core dependencies installed successfully!")
print(f"   - tiktoken: {tiktoken.__version__}")
print(f"   - datasets: {datasets.__version__}")
print(f"   - fastapi: {fastapi.__version__}")

## 0.3 Build the Rust Tokenizer (Optional but Recommended)

Nanochat uses a custom Rust-based BPE tokenizer for fast training. Let's build it.

In [ ]:
# Install Rust (if not already installed)
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ['PATH'] = f"{os.environ['HOME']}/.cargo/bin:{os.environ['PATH']}"

# Verify Rust installation
!rustc --version
!cargo --version

In [ ]:
# Install maturin (Python-Rust bridge)
!pip install maturin -q

# Build the Rust tokenizer
%cd rustbpe
!maturin develop --release
%cd ..

# Verify rustbpe installation
import rustbpe
print(f"✅ RustBPE installed successfully!")

## 0.4 Project Structure Overview

Let's explore the nanochat codebase structure.

In [ ]:
# Show project structure
!find . -type f -name "*.py" | head -30
print("\n" + "="*60)
!wc -l nanochat/*.py scripts/*.py tasks/*.py 2>/dev/null | tail -1

### Key Directories

```
nanochat/
├── nanochat/              # Core library (~5K lines)
│   ├── gpt.py            # GPT model architecture
│   ├── engine.py         # Inference engine with KV cache
│   ├── tokenizer.py      # BPE tokenizer wrapper
│   ├── dataloader.py     # Distributed data loading
│   ├── muon.py           # Muon optimizer
│   └── ...
│
├── scripts/               # Training & inference scripts
│   ├── base_train.py     # Pre-training
│   ├── chat_sft.py       # Supervised fine-tuning
│   ├── chat_web.py       # Web server
│   └── ...
│
├── tasks/                 # Evaluation benchmarks
│   ├── gsm8k.py          # Math reasoning
│   ├── mmlu.py           # Knowledge
│   └── ...
│
├── rustbpe/               # Rust tokenizer
└── learning_guide/        # This tutorial series!
```

## 0.5 Quick Sanity Check

Let's run a quick test to make sure everything is working.

In [ ]:
import torch
import torch.nn.functional as F

# Test basic PyTorch operations on GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create a simple tensor operation
x = torch.randn(1000, 1000, device=device)
y = torch.randn(1000, 1000, device=device)

# Matrix multiplication
import time
torch.cuda.synchronize() if device.type == 'cuda' else None
t0 = time.time()
for _ in range(100):
    z = torch.mm(x, y)
torch.cuda.synchronize() if device.type == 'cuda' else None
t1 = time.time()

print(f"100 matrix multiplications (1000x1000): {(t1-t0)*1000:.2f}ms")
print(f"TFLOPS estimate: {100 * 2 * 1000**3 / (t1-t0) / 1e12:.2f}")

# Test bfloat16 support (important for training)
if device.type == 'cuda':
    x_bf16 = x.to(torch.bfloat16)
    y_bf16 = y.to(torch.bfloat16)
    z_bf16 = torch.mm(x_bf16, y_bf16)
    print(f"✅ bfloat16 supported!")

print("\n✅ All sanity checks passed!")

## 0.6 Memory Management Tips for Colab

When training large models, memory management is crucial. Here are some tips:

In [ ]:
def print_gpu_memory():
    """Print current GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        max_allocated = torch.cuda.max_memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"GPU Memory:")
        print(f"  - Allocated: {allocated:.2f} GB")
        print(f"  - Reserved:  {reserved:.2f} GB")
        print(f"  - Max Allocated: {max_allocated:.2f} GB")
        print(f"  - Total: {total:.2f} GB")
        print(f"  - Free: {total - reserved:.2f} GB")
    else:
        print("No GPU available")

def clear_gpu_memory():
    """Clear GPU memory cache."""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    print("✅ GPU memory cache cleared")

# Show current memory usage
print_gpu_memory()

### Memory Tips

1. **Use gradient checkpointing** for training large models
2. **Use bfloat16** precision (default in nanochat)
3. **Clear cache between experiments** using `clear_gpu_memory()`
4. **Monitor memory** regularly using `print_gpu_memory()`
5. **Reduce batch size** if you encounter OOM errors

## 0.7 Understanding CUDA Streams (Preview)

For efficient GPU utilization, it's important to understand CUDA streams. Here's a quick preview:

In [ ]:
if torch.cuda.is_available():
    # Default stream
    default_stream = torch.cuda.current_stream()
    print(f"Default stream: {default_stream}")
    
    # Create a new stream
    stream = torch.cuda.Stream()
    print(f"New stream: {stream}")
    
    # Operations in different streams can run concurrently
    x = torch.randn(1000, 1000, device='cuda')
    
    with torch.cuda.stream(stream):
        # This runs asynchronously on the new stream
        y = x @ x.T
    
    # Synchronize if needed
    stream.synchronize()
    print("✅ CUDA streams working correctly!")
else:
    print("CUDA streams require a GPU")

## Summary

In this notebook, we:

1. ✅ Verified GPU availability and specifications
2. ✅ Cloned and installed the nanochat repository
3. ✅ Built the Rust tokenizer (RustBPE)
4. ✅ Explored the project structure
5. ✅ Ran sanity checks for PyTorch and bfloat16
6. ✅ Learned memory management tips

## Next Steps

Continue to **[Module 1: BPE Tokenizer](01_bpe_tokenizer.ipynb)** to learn:
- How Byte Pair Encoding (BPE) works
- Training a tokenizer from scratch
- Efficient tokenization with tiktoken

---

**Estimated time for this notebook: 15-20 minutes**